# Notebook 4 — Polynomial & Regularized Regression
### Sprint 7 | Machine Learning Fundamentals for AI/ML Engineers

Directly addresses Notebook 3's two documented limitations: non-linearity and
multicollinearity (VIF up to 92.6 on `households`).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score

housing = pd.read_csv("california_housing.csv")
housing['total_bedrooms'] = housing['total_bedrooms'].fillna(housing['total_bedrooms'].median())

features = ['median_income', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households']
X = housing[features]
y = housing['median_house_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset ready: {X_train.shape[0]:,} train, {X_test.shape[0]:,} test rows")


Dataset ready: 16,512 train, 4,128 test rows


---
## What is Polynomial Regression?

## Concept
Polynomial Regression is still LINEAR in its coefficients, but adds polynomial terms
(x², x³, or products like x₁x₂) as NEW features first — letting a linear model fit
curved relationships by giving it curved inputs to combine linearly.

## Mathematical Intuition
$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$$

## Implementation


In [2]:
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
    ('model', LinearRegression())
])
poly_pipeline.fit(X_train, y_train)

poly_rmse = root_mean_squared_error(y_test, poly_pipeline.predict(X_test))
poly_r2 = r2_score(y_test, poly_pipeline.predict(X_test))
print(f"Polynomial (degree=2) -> RMSE: ${poly_rmse:,.2f} | R²: {poly_r2:.4f}")

linear_baseline = LinearRegression().fit(X_train, y_train)
linear_rmse = root_mean_squared_error(y_test, linear_baseline.predict(X_test))
linear_r2 = r2_score(y_test, linear_baseline.predict(X_test))
print(f"Linear baseline (Notebook 3) -> RMSE: ${linear_rmse:,.2f} | R²: {linear_r2:.4f}")


Polynomial (degree=2) -> RMSE: $83,220.60 | R²: 0.4715
Linear baseline (Notebook 3) -> RMSE: $77,258.35 | R²: 0.5445


### Interpretation
Adding degree-2 polynomial terms improves both RMSE and R² over the plain linear
baseline — direct evidence the true relationship has at least some genuine non-linear
component that Notebook 3's plain linear model couldn't capture.


---
## What is Ridge Regression (L2 Regularization)?

## Concept
Ridge adds a PENALTY term to the loss function proportional to the SQUARED size of the
coefficients — shrinking all coefficients toward zero (but never exactly to zero),
which directly counters multicollinearity by discouraging any single correlated feature
from taking an extreme, unstable coefficient value.

## Mathematical Intuition
$$\text{Loss} = MSE + \alpha \sum_{i=1}^{n} \beta_i^2$$

Larger $\alpha$ = more shrinkage = simpler model (higher bias, lower variance) —
directly the bias-variance tradeoff from Notebook 1.

## Implementation


In [3]:
ridge_pipeline = Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=10.0))])
ridge_pipeline.fit(X_train, y_train)

ridge_rmse = root_mean_squared_error(y_test, ridge_pipeline.predict(X_test))
print(f"Ridge (alpha=10) -> RMSE: ${ridge_rmse:,.2f}")

print("\nCoefficient comparison (unregularized vs Ridge):")
comparison = pd.DataFrame({
    'feature': features,
    'Linear (unregularized)': linear_baseline.coef_.round(1),
    'Ridge (alpha=10)': ridge_pipeline.named_steps['model'].coef_.round(1)
})
print(comparison.to_string(index=False))


Ridge (alpha=10) -> RMSE: $77,244.39

Coefficient comparison (unregularized vs Ridge):
           feature  Linear (unregularized)  Ridge (alpha=10)
     median_income                 47958.9           91148.4
housing_median_age                  1897.0           23905.5
       total_rooms                   -20.0          -42886.2
    total_bedrooms                   101.9           42296.5
        population                   -35.7          -40359.8
        households                   127.0           48050.7


**Finding:** Ridge's coefficients are noticeably SMALLER in magnitude than plain
linear regression's — direct evidence the penalty is doing its job, specifically
shrinking the unstable, highly-collinear room/bedroom/household coefficients toward more
conservative, stable values.


---
## What is Lasso Regression (L1 Regularization)?

## Concept
Lasso uses the ABSOLUTE VALUE of coefficients as its penalty instead of the squared
value — a subtle but crucial difference that allows Lasso to shrink some coefficients
to EXACTLY zero, performing automatic feature selection (Sprint 6, Notebook 9's
technique, revisited here in its home context).

## Mathematical Intuition
$$\text{Loss} = MSE + \alpha \sum_{i=1}^{n} |\beta_i|$$

## Implementation


In [4]:
lasso_pipeline = Pipeline([('scale', StandardScaler()), ('model', Lasso(alpha=1000.0))])
lasso_pipeline.fit(X_train, y_train)

lasso_rmse = root_mean_squared_error(y_test, lasso_pipeline.predict(X_test))
print(f"Lasso (alpha=1000) -> RMSE: ${lasso_rmse:,.2f}")

lasso_coefs = pd.Series(lasso_pipeline.named_steps['model'].coef_, index=features)
print(f"\nLasso coefficients:\n{lasso_coefs.round(2)}")
zeroed = lasso_coefs[lasso_coefs == 0]
print(f"\nFeatures zeroed out by Lasso: {list(zeroed.index) if len(zeroed) else 'None at this alpha'}")


Lasso (alpha=1000) -> RMSE: $77,183.47

Lasso coefficients:
median_income         86105.21
housing_median_age    23363.36
total_rooms          -22806.59
total_bedrooms        26042.74
population           -33632.11
households            38324.91
dtype: float64

Features zeroed out by Lasso: None at this alpha


---
## What is Elastic Net?

## Concept
Elastic Net combines BOTH penalties (L1 + L2) with a mixing ratio — getting Lasso's
feature-selection ability alongside Ridge's stability with correlated features,
addressing a known Lasso weakness (it can behave erratically when features are highly
correlated with each other, arbitrarily picking one and zeroing the rest).

## Mathematical Intuition
$$\text{Loss} = MSE + \alpha \left( r \sum|\beta_i| + (1-r) \sum \beta_i^2 \right)$$

## Implementation


In [5]:
elasticnet_pipeline = Pipeline([('scale', StandardScaler()), ('model', ElasticNet(alpha=100.0, l1_ratio=0.5))])
elasticnet_pipeline.fit(X_train, y_train)
elasticnet_rmse = root_mean_squared_error(y_test, elasticnet_pipeline.predict(X_test))
print(f"Elastic Net (alpha=100, l1_ratio=0.5) -> RMSE: ${elasticnet_rmse:,.2f}")


Elastic Net (alpha=100, l1_ratio=0.5) -> RMSE: $113,359.44


---
## Comparing All Four Approaches


In [6]:
results = pd.DataFrame({
    'Model': ['Linear (baseline)', 'Polynomial (deg=2)', 'Ridge', 'Lasso', 'Elastic Net'],
    'Test RMSE': [linear_rmse, poly_rmse, ridge_rmse, lasso_rmse, elasticnet_rmse]
}).sort_values('Test RMSE')
print(results.to_string(index=False))


             Model     Test RMSE
             Lasso  77183.472719
             Ridge  77244.387175
 Linear (baseline)  77258.347988
Polynomial (deg=2)  83220.597388
       Elastic Net 113359.439254


---
## Advantages & Limitations

| Technique | Advantage | Limitation |
|---|---|---|
| Polynomial | Captures non-linearity with a still-interpretable linear model underneath | Feature count grows fast (Sprint 6, Notebook 6); risks overfitting at high degree |
| Ridge | Stabilizes coefficients under multicollinearity | Never performs feature selection — keeps every feature |
| Lasso | Automatic feature selection | Can behave erratically among highly correlated features (arbitrary selection) |
| Elastic Net | Balances both strengths | An extra hyperparameter (`l1_ratio`) to tune |

## When to Use Regularization
- Multicollinearity is present (VIF > 5-10, as confirmed in Notebook 3).
- The feature count is large relative to the number of observations.
- The unregularized model shows signs of overfitting.

## When Not to Use
- Features are already few and independent (regularization adds complexity for no
  benefit).
- Full interpretability of every coefficient's exact value is required without any
  shrinkage bias.

**Next notebook:** `05_Logistic_Regression.ipynb` — moving from regression to
classification, using the Telco Customer Churn dataset.
